# GX Essentials - Companion Notebook - Solutions

This is a companion notebook for the course containing all code shown in the course as well as solutions to the challenge. 

Feel free to reference this code where needed.

### Creating a GX Data Context

In [ ]:
import great_expectations as gx

In [ ]:
print(gx.__version__)

In [ ]:
my_context = gx.get_context(mode="file")

### Creating a Data Source

In [ ]:
my_connection_string = "${POSTGRES_CONNECTION_STRING}"

In [ ]:
my_data_source = my_context.data_sources.add_postgres(
    name="taxi_data_postgres", connection_string=my_connection_string
)

### Creating a Data Asset

In [ ]:
my_data_asset = my_data_source.add_table_asset(
    table_name="yellow_tripdata_2025_01", 
    schema_name="taxidata",
    name="taxi_jan2025_table"
)

In [ ]:
print(my_data_source.assets)

### Creating a Batch Definition

In [ ]:
my_batch_definition = my_data_asset.add_batch_definition_whole_table(
    name="FULL_TABLE"
)

In [ ]:
my_batch = my_batch_definition.get_batch()

In [ ]:
my_batch.head()

### Defining Expectations

In [ ]:
my_notnull_expectation = gx.expectations.ExpectColumnValuesToNotBeNull(
    column="tpep_pickup_datetime"
)

In [ ]:
my_suite = gx.ExpectationSuite(name="my_expectation_suite")

In [ ]:
my_suite.add_expectation(my_notnull_expectation)

In [ ]:
my_context.suites.add(my_suite)

### Running Validations

In [ ]:
my_validation_results = my_batch.validate(my_notnull_expectation)

In [ ]:
print(my_validation_results)

In [ ]:
my_validation_definition = gx.ValidationDefinition(
    data=my_batch_definition, 
    suite=my_suite, 
    name="my_validation_definition"
)

In [ ]:
my_context.validation_definitions.add(my_validation_definition)

In [ ]:
my_validation_results = my_validation_definition.run()

In [ ]:
print(my_validation_results)

### Using Data Docs

In [ ]:
# We can access the docs in a browser at http://localhost:8080/local_site/
# This URL is different from the output due to the Docker container configuration

my_context.build_data_docs()

### Challenge: Connect & run GX - Solution

In [ ]:
my_data_asset_feb = my_data_source.add_table_asset(
    table_name="yellow_tripdata_2025_02", 
    schema_name="taxidata",
    name="taxi_feb2025_table"
)

my_batch_definition_feb = my_data_asset_feb.add_batch_definition_whole_table(
    name="FULL_TABLE"
)

my_batch_feb = my_batch_definition_feb.get_batch()

In [ ]:
my_passengercount_expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="passenger_count", 
    min_value=1
)

my_suite.add_expectation(my_passengercount_expectation)
my_context.suites.add_or_update(my_suite)

In [ ]:
my_validation_results = my_validation_definition.run()

In [ ]:
print(my_validation_results)

In [ ]:
# Optional: Build Data Docs and open them at http://localhost:8080/local_site/
my_context.build_data_docs()

### Triggering actions with Checkpoints

In [ ]:
my_action_list = [
    gx.checkpoint.UpdateDataDocsAction(name="update_all_data_docs")
]

In [ ]:
my_validation_definitions = [
    my_context.validation_definitions.get("my_validation_definition")
]

In [ ]:
my_checkpoint = gx.Checkpoint(
    name="my_checkpoint",
    validation_definitions=my_validation_definitions,
    actions=my_action_list
)

In [ ]:
my_context.checkpoints.add(my_checkpoint)

In [ ]:
my_checkpoint.run()

### Root cause analysis of test failures

In [ ]:
# Creating an additional failing Expectation here - this isn't shown in the video

my_positive_amount_expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="total_amount", 
    min_value=0.0,
)
my_suite.add_expectation(my_positive_amount_expectation)
my_context.suites.add_or_update(my_suite)
my_context.checkpoints.get("my_checkpoint").run()

In [ ]:
# Load the source data parquet file into a Pandas dataframe

import pandas as pd
df = pd.read_parquet("/app/taxi-data/yellow_tripdata_2025-01.parquet")
df

In [ ]:
# Group by passenger_count and count how often each value occurs

df.groupby('passenger_count').size()

In [ ]:
# Get the min and max values in the total_amount column

df['total_amount'].describe()[['min', 'max']]

### Creating Fuzzy Expectations

In [ ]:
my_fuzzy_passengercount_expectation = gx.expectations.ExpectColumnValuesToBeBetween(
    column="passenger_count", 
    min_value=1,
    mostly=0.99
)

### Updating and deleting Expectations in an Expectation Suite

In [ ]:
my_suite.add_expectation(my_fuzzy_passengercount_expectation)

In [ ]:
my_context.suites.all()

In [ ]:
my_suite.remove_expectation(id="PASTE EXPECTATION ID HERE")

In [ ]:
my_context.suites.add_or_update(my_suite)

In [ ]:
my_checkpoint = my_context.checkpoints.get("my_checkpoint")

In [ ]:
my_checkpoint.run()

### Creating custom Expectations

In [ ]:
my_query = "SELECT * FROM {batch} WHERE tpep_dropoff_datetime < tpep_pickup_datetime"

In [ ]:
my_custom_expectation = gx.expectations.UnexpectedRowsExpectation(
    unexpected_rows_query=my_query
)

In [ ]:
my_suite.add_expectation(my_custom_expectation)
my_context.suites.add_or_update(my_suite)

In [ ]:
my_checkpoint = my_context.checkpoints.get("my_checkpoint")
my_checkpoint.run()

**This is the end of the companion notebook. Congratulations for completing the course!**